## Подробное исследование признаков

В данном ноутбуке представлена история исследования данных и введение правил обработки.

Загрузка необходимых библиотек и исходных данных

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from src.utils import evaluate_preds, show_metrics
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from catboost import CatBoostClassifier, CatBoostRegressor

df = pd.read_csv('../data/raw/train.csv')

Начнем обработку категориальных признаков. Рассмотрим `Home Ownership`. Он может принимать значение `Have Mortgage`, оно равнозначно по смыслу со значением `Home Mortgage`. Сформулируем правило исправления:
1. `Home Ownerhip = Home Mortgage` при `Home Ownership = Have Mortgage`

In [ ]:
home_ownership_is_have_morgage_mask = (df['Home Ownership'] == 'Have Mortgage')
df.loc[home_ownership_is_have_morgage_mask, 'Home Ownership'] = 'Home Mortgage'

Рассмотрим категориальный признак `Years in current job`. Он принимает строковые значения: `< 1 year`, `1 year` ... `10 years`, `10+ years` и имеет пропущенные значения. Опишем порядок преобразования строковых значений в численное:
1. Значения `< 1 year` и `10+ years` изменяем на `1 year` и `10 years` соответственно. Оставляем флаги `less_1_year_in_current_job` и `more_10_years_in_current_job`
2. Применяем функцию `split` и преобразуем первый элемент к целочисленному значению

К выбору метода для заполнения пропусков вернемся после очистки выбросов остальных признаков

In [ ]:
yicj_less_1_year_mask = (df['Years in current job'] == '< 1 year')
yicj_more_10_years_mask = (df['Years in current job'] == '10+ years')

df['less_1_year_in_current_job'] = yicj_less_1_year_mask * 1
df['more_10_years_in_current_job'] = yicj_more_10_years_mask * 1

df.loc[yicj_less_1_year_mask, 'Years in current job'] = '1 year'
df.loc[yicj_more_10_years_mask, 'Years in current job'] = '10 years'

df['Years in current job'] = df['Years in current job'].apply(lambda g: int(g.split(' ')[0]) if isinstance(g, str) else g)

Оставшиеся категориальные признаки не имеют пропусков и не требуют коррекций. Заменим все категории дамми-переменными

In [ ]:
df = pd.get_dummies(data=df, prefix=None, prefix_sep='_', columns=['Home Ownership', 'Purpose', 'Term'], drop_first=True, dtype=int)
df.head()

Рассмотрим признак `Annual Income`. Рассмотрим экстремальные значения `Annual Income > Q_99` (где Q_99 квартиль соответствующего признака)

In [ ]:
annual_income_q99 = df['Annual Income'].quantile(0.99)
annual_income_gt_q99_mask = (df['Annual Income'] >= annual_income_q99)
display(df[annual_income_gt_q99_mask])
sns.displot(df['Annual Income'], bins=100)

Не будем корректировать высокие значения, так как распределение выглядит вполне естественно (нормальное с правым хвостом) для признака финансовой природы.

Сформулируем идею для добавления новых признаков: добавить признак `log_income = log(Annual income)`

Рассмотрим признак `Number of Open Accounts`. Правило обработки:
1. Значения `Number of open accounts > Q_99` будем считать выбросами и заменять медианой. Оставим флаг `number_of_open_accounts_was_scaled`

In [ ]:
noa_q99 = df['Number of Open Accounts'].quantile(0.99)
noa_greater_q99 = (df['Number of Open Accounts'] > noa_q99)
df['number_of_open_accounts_was_scaled'] = noa_greater_q99
df.loc[noa_greater_q99, 'Number of Open Accounts'] = df['Number of Open Accounts'].median()

Рассмотрим признак `Maximum Open Credit`. Правило обработки:
1. `Maximum Open Credit = median` для `Maximum Open Credit > 0.15e9`
2. `Maximum Open Credit = median` для `Maximum Open Credit = 0`

In [ ]:
moc_big_value_mask = (df['Maximum Open Credit'] > 1e7) & (df['Maximum Open Credit'].notna())
df.loc[moc_big_value_mask, 'Maximum Open Credit'] = df['Maximum Open Credit'].median()
moc_0_mask = (df['Maximum Open Credit'] == 0)
df.loc[moc_0_mask, 'Maximum Open Credit'] = df['Maximum Open Credit'].median()

Рассмотрим признак `Current Loan Amount`. У всех объектов, у которых `Current Loan Amount = 99999999` значение `Credit Default = 0`. Оставим флаг для этих значений и заменим медианой.

In [ ]:
cla_scaled_mask = df['Current Loan Amount'] == 99999999.0
display(df.loc[cla_scaled_mask, 'Credit Default'].value_counts())

df['current_loan_amount_scaled'] = cla_scaled_mask * 1
df.loc[cla_scaled_mask, 'Current Loan Amount'] = df['Current Loan Amount'].median()

Рассмотрим признак `Monthly Dept`. Правило для коррекции:
1. `Monthly Dept = Current Loan Amount` при `Monthly Dept > Current Loan Amount` (Для погашения займа осталась одна выплата)

In [ ]:
monthly_debt_gt_cla_mask = df['Monthly Debt'] > df['Current Loan Amount']
df.loc[monthly_debt_gt_cla_mask, 'Monthly Debt'] = df.loc[monthly_debt_gt_cla_mask, 'Current Loan Amount']

Рассмотрим признак `Credit Score`. Заметим, что большинство объектов имеют `Credit Score` в диапозоне от 585 до 751, а все значения, первышающие 751 оканчиваются на 0. Можно сделать вывод, что большие значения (> 751) носят не случайный характер. Все объекты `Credit Score` которых превышает 751 имеют `Credit Default` 1. Правила для исправления:
1. Оставляем флаг `credit_score_scaled`
2. `Credit Score = Credit Score // 10` для объектов у которых `Credit Score > 751`.  

In [ ]:
print(np.sort(df.loc[df['Credit Score'] < 1000, 'Credit Score'].dropna().unique()))
print(np.sort(df.loc[df['Credit Score'] > 1000, 'Credit Score'].dropna().unique()))
display(df.loc[df['Credit Score'] > 751, 'Credit Default'].value_counts())

credit_score_scaled_mask = (df['Credit Score'] > 751)
df['credit_score_scaled'] = credit_score_scaled_mask
df.loc[credit_score_scaled_mask, 'Credit Score'] = df.loc[credit_score_scaled_mask, 'Credit Score'] // 10

## Исследование пропусков в данных

Рассмотри пропуски в признаке `Months since last delinquent`. Используемое правило заполнения:
1. `Months since last delinquent = 0` для объектов у которых `Number of Credit Problems > 0` (неуплата в текущем месяце)
2. `Months since last delinqunt = -1` + `not_debtor` для объектов, у которых `Number of Credit Problems = 0` (не было пропущенных платежей)

In [ ]:
msld_isna_nocp_1 = df['Months since last delinquent'].isna() & (df['Number of Credit Problems'] >= 1)
msld_isna_nocp_0 = df['Months since last delinquent'].isna() & (df['Number of Credit Problems'] == 0)
df['not_debtor'] = msld_isna_nocp_0
df.loc[msld_isna_nocp_1, 'Months since last delinquent'] = 0
df.loc[msld_isna_nocp_0, 'Months since last delinquent'] = -1

In [ ]:
def yicj_median_imputer(df):
    X = df.copy()
    train_mask = X['Years in current job'].notna()
    X = X[train_mask]
    y = X['Years in current job']
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.25)
    median_imputer = np.ones_like(y_test) * y_train.median() 
    show_metrics(y_test, median_imputer, average='macro')

def yicj_knn_imputer(df):
    X = df.copy()
    train_mask = X['Years in current job'].notna()
    X = X[train_mask].dropna()
    y = X['Years in current job']
    X.drop(columns=['Years in current job', 'Credit Default'], inplace=True)
    scaler = RobustScaler()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    knn_imputer = KNeighborsClassifier(n_neighbors=6)
    knn_imputer.fit(X_train, y_train)
    y_pred = knn_imputer.predict(X_test)
    y_pred_proba = knn_imputer.predict_proba(X_test)[:, 1]
    show_metrics(y_test, y_pred, y_pred_proba, average='weighted')

def yicj_catboost_imputer(df):
    X = df.copy()
    train_mask = X['Years in current job'].notna()
    X = X[train_mask].dropna()
    y = X['Years in current job']
    X.drop(columns=['Years in current job', 'Credit Default'], inplace=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    frozen_params = {
        'silent':True,
        #'cat_features':CAT_FEATURE_NAMES,
        'eval_metric':'MultiClass',
        'early_stopping_rounds':20,
        'random_state':42
    }
    catboost_imputer = CatBoostClassifier(**frozen_params)
    catboost_imputer.fit(X_train, y_train)
    y_pred = catboost_imputer.predict(X_test)
    y_pred_proba = catboost_imputer.predict_proba(X_test)[:, 1]
    show_metrics(y_test, y_pred, y_pred_proba, average='weighted')
    return catboost_imputer


yicj_median_imputer(df)
print('#'*20)
yicj_knn_imputer(df)
print('#'*20)
catboost_imputer_yicj = yicj_catboost_imputer(df)

Даже без подобранных гиперпараметров, catboost справляется c заполнением лучше остальных предложенных методов. Будем использовать его как основной. Подбор гиперпараметров реализуем в transformers.py

In [ ]:
yicj_isna_mask = df['Years in current job'].isna()
temp = df[yicj_isna_mask].drop(columns=['Years in current job', 'Credit Default'])
df.loc[yicj_isna_mask, 'Years in current job'] = catboost_imputer_yicj.predict(temp)

Аналогичный тест проведем и для признака `Bankruptcies`

In [ ]:
def bankrupties_median_imputer(df):
    X = df.copy()
    train_mask = X['Bankruptcies'].notna()
    X = X[train_mask]
    y = X['Bankruptcies']
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.25)
    median_imputer = np.ones_like(y_test) * y_train.median() 
    show_metrics(y_test, median_imputer, average='macro')

def bankrupties_knn_imputer(df):
    X = df.copy()
    train_mask = X['Bankruptcies'].notna()
    X = X[train_mask].dropna()
    y = X['Bankruptcies']
    X.drop(columns=['Bankruptcies', 'Credit Default'], inplace=True)
    scaler = RobustScaler()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    knn_imputer = KNeighborsClassifier(n_neighbors=6)
    knn_imputer.fit(X_train, y_train)
    y_pred = knn_imputer.predict(X_test)
    y_pred_proba = knn_imputer.predict_proba(X_test)[:, 1]
    show_metrics(y_test, y_pred, y_pred_proba, average='weighted')

def bankrupties_catboost_imputer(df):
    X = df.copy()
    train_mask = X['Bankruptcies'].notna()
    X = X[train_mask].dropna()
    y = X['Bankruptcies']
    X.drop(columns=['Bankruptcies', 'Credit Default'], inplace=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    frozen_params = {
        'silent':True,
        #'cat_features':CAT_FEATURE_NAMES,
        'eval_metric':'MultiClass',
        'early_stopping_rounds':20,
        'random_state':42
    }
    catboost_imputer = CatBoostClassifier(**frozen_params)
    catboost_imputer.fit(X_train, y_train)
    y_pred = catboost_imputer.predict(X_test)
    y_pred_proba = catboost_imputer.predict_proba(X_test)[:, 1]
    show_metrics(y_test, y_pred, y_pred_proba, average='weighted')
    return catboost_imputer


bankrupties_median_imputer(df)
print('#'*20)
bankrupties_knn_imputer(df)
print('#'*20)
bankrupties_catboost_imputer(df)

In [ ]:
def credit_score_median_imputer(df):
    X = df.copy()
    train_mask = X['Credit Score'].notna()
    X = X[train_mask]
    y = X['Credit Score']
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.25)
    median_imputer = np.ones_like(y_test) * y_train.median() 
    evaluate_preds(y_test, median_imputer)

def credit_score_knn_imputer(df):
    X = df.copy()
    train_mask = X['Credit Score'].notna()
    X = X[train_mask].dropna()
    y = X['Credit Score']
    X.drop(columns=['Credit Score', 'Credit Default'], inplace=True)
    scaler = RobustScaler()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    knn_imputer = KNeighborsRegressor(n_neighbors=6)
    knn_imputer.fit(X_train, y_train)
    y_pred = knn_imputer.predict(X_test)
    evaluate_preds(y_test, y_pred)

def credit_score_catboost_imputer(df):
    X = df.copy()
    train_mask = X['Credit Score'].notna()
    X = X[train_mask].dropna()
    y = X['Credit Score']
    X.drop(columns=['Credit Score', 'Credit Default'], inplace=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    frozen_params = {
        'silent':True,
        #'cat_features':CAT_FEATURE_NAMES,
        'eval_metric':'RMSE',
        'early_stopping_rounds':20,
        'random_state':42
    }
    catboost_imputer = CatBoostRegressor(**frozen_params)
    catboost_imputer.fit(X_train, y_train)
    y_pred = catboost_imputer.predict(X_test)
    evaluate_preds(y_test, y_pred)
    return catboost_imputer


credit_score_median_imputer(df)
print('#'*20)
credit_score_knn_imputer(df)
print('#'*20)
credit_score_catboost_imputer(df)

In [ ]:
def annual_income_median_imputer(df):
    X = df.copy()
    train_mask = X['Annual Income'].notna()
    X = X[train_mask]
    y = X['Annual Income']
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.25)
    median_imputer = np.ones_like(y_test) * y_train.median() 
    evaluate_preds(y_test, median_imputer)

def annual_income_knn_imputer(df):
    X = df.copy()
    train_mask = X['Annual Income'].notna()
    X = X[train_mask].dropna()
    y = X['Annual Income']
    X.drop(columns=['Annual Income', 'Credit Default'], inplace=True)
    scaler = RobustScaler()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    knn_imputer = KNeighborsRegressor(n_neighbors=6)
    knn_imputer.fit(X_train, y_train)
    y_pred = knn_imputer.predict(X_test)
    evaluate_preds(y_test, y_pred)

def annual_income_catboost_imputer(df):
    X = df.copy()
    train_mask = X['Annual Income'].notna()
    X = X[train_mask].dropna()
    y = X['Annual Income']
    X.drop(columns=['Annual Income', 'Credit Default'], inplace=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    frozen_params = {
        'silent':True,
        #'cat_features':CAT_FEATURE_NAMES,
        'eval_metric':'RMSE',
        'early_stopping_rounds':20,
        'random_state':42
    }
    catboost_imputer = CatBoostRegressor(**frozen_params)
    catboost_imputer.fit(X_train, y_train)
    y_pred = catboost_imputer.predict(X_test)
    evaluate_preds(y_test, y_pred)
    return catboost_imputer

def median_by_job(df):
    X = df.copy()
    train_mask = X['Annual Income'].notna()
    X = X[train_mask].dropna()
    y = X['Annual Income']
    X.drop(columns=['Credit Default'], inplace=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    temp = pd.DataFrame()
    temp = X_train[['Years in current job', 'Annual Income']]
    median_by_job = temp.groupby('Years in current job')['Annual Income'].median()
    y_pred = X_test['Years in current job'].map(median_by_job)
    evaluate_preds(y_test, y_pred)



annual_income_median_imputer(df)
print('#'*20)
annual_income_knn_imputer(df)
print('#'*20)
annual_income_catboost_imputer(df)
median_by_job(df)

Порядок преобразований:
1. `Home Ownerhip = Home Mortgage` для `Home Ownership = Have Mortgage`
2. Значения `< 1 year` и `10+ years` изменяем на `1 year` и `10 years` соответственно. Оставляем флаги `less_1_year_in_current_job` и `more_10_years_in_current_job`
3. Значения `Number of open accounts > Q_99` будем считать выбросами и заменять медианой. Оставим флаг `number_of_open_accounts_was_scaled`
4.  `Maximum Open Credit = median` для `Maximum Open Credit > 0.15e9` и  `Maximum Open Credit = 0`
5. `Current Loan Amount = median` для `Current Loan Amount = 99999999` + флаг `current_loan_amount_scaled`
6. `Monthly Dept = Current Loan Amount` для `Monthly Dept > Current Loan Amount`
7. `Credit Score = Credit Score // 10` для `Credit Score > 751` + флаг `credit_score_scaled`
8. `Months since last delinquent = 0` для `Number of Credit Problems > 0` 
9. `Months since last delinqunt = -1` + флаг `not_debtor` для `Number of Credit Problems = 0`
10. Заполнение `Years in current job` при помощи `CatBoost` + флаг `years_in_current_job_filled`
11. Заполнение `Bankrupties` при помощи `CatBoost` + флаг `bankrupties_filled`
12. Заполнение `Credit Score` при помощи `CatBoost` + флаг `credit_score_filled`
13. Заполнение `Annual Income` при помощи `CatBoost` + флаг `annual_income_filled`